# Chess Engine Scenario Scoring

This notebook scores models from `summary.csv` and builds a leaderboard using your chosen tradeoffs.

In [1]:
import numpy as np

In [5]:
from pathlib import Path
import pandas as pd

SUMMARY_PATH = Path(r"C:\Users\etu89\OneDrive\Documents\CubistHack\CubistChessEngine\scenario_outputs/run_20260424_235230\summary.csv")
print("Exists:", SUMMARY_PATH.exists(), SUMMARY_PATH)
df = pd.read_csv(SUMMARY_PATH)
df.head()



Exists: True C:\Users\etu89\OneDrive\Documents\CubistHack\CubistChessEngine\scenario_outputs\run_20260424_235230\summary.csv


,model,round,stated_evaluation_function,chosen_rewrite,reason_for_rewrite,tests_passed,tests_failed,win_rate_vs_baseline,win_rate_vs_previous_self,win_rate_vs_other_models,illegal_move_count,average_move_time_ms
0,classical_minimax,1,classical_minimax default placeholder (no impr...,No automatic rewrite executed,Improver command not configured for this model...,1,0,0.500,0.5,0.78125,5,226.315085
1,claude_api,1,claude_api default placeholder (no improver co...,No automatic rewrite executed,Improver command not configured for this model...,0,0,0.000,0.0,0.00000,28,0.000000
2,monte_carlo,1,monte_carlo default placeholder (no improver c...,No automatic rewrite executed,Improver command not configured for this model...,0,1,0.125,0.5,0.25000,5,705.208264
3,mock_nn,1,mock_nn default placeholder (no improver comma...,No automatic rewrite executed,Improver command not configured for this model...,1,0,1.000,0.5,1.00000,5,881.090805
4,berserker1,1,berserker1 default placeholder (no improver co...,No automatic rewrite executed,Improver command not configured for this model...,0,1,0.750,0.5,0.68750,5,765.268808


In [6]:
# Aggregate per model
agg = (
    df.groupby("model", as_index=False)
      .agg(
          rounds=("round", "max"),
          mean_vs_others=("win_rate_vs_other_models", "mean"),
          mean_vs_baseline=("win_rate_vs_baseline", "mean"),
          mean_vs_prev=("win_rate_vs_previous_self", "mean"),
          total_illegal=("illegal_move_count", "sum"),
          total_tests_failed=("tests_failed", "sum"),
          mean_move_time_ms=("average_move_time_ms", "mean"),
      )
)

def slope_vs_prev(sub_df: pd.DataFrame) -> float:
    if len(sub_df) < 2:
        return 0.0
    x = sub_df["round"].to_numpy(dtype=float)
    y = sub_df["win_rate_vs_previous_self"].to_numpy(dtype=float)
    return float(np.polyfit(x, y, 1)[0])

slopes = (
    df.sort_values(["model", "round"])
      .groupby("model")
      .apply(slope_vs_prev)
      .rename("improvement_slope")
      .reset_index()
)

scored = agg.merge(slopes, on="model", how="left")
scored

,model,rounds,mean_vs_others,mean_vs_baseline,mean_vs_prev,total_illegal,total_tests_failed,mean_move_time_ms,improvement_slope
0,berserker1,5,0.68750,0.750,0.500,25,5,772.702469,-1.206738e-17
1,classical_minimax,5,0.73125,0.500,0.500,25,4,246.000302,-1.206738e-17
2,claude_api,5,0.00000,0.000,0.000,140,0,0.000000,0.000000e+00
3,mock_nn,5,1.00000,1.000,0.500,25,0,866.276932,-1.206738e-17
4,monte_carlo,5,0.25000,0.075,0.475,27,5,692.011593,1.250000e-02


In [7]:
# Scoring knobs (edit these to match your priorities)
WEIGHTS = {
    "vs_others": 0.45,
    "vs_baseline": 0.35,
    "vs_prev": 0.20,
}

PENALTIES = {
    "illegal_per_move": 0.10,   # subtract this much per illegal move
    "failed_test": 0.25,        # subtract this much per failed test
    "slow_ms_scale": 0.0002,    # subtract scale * mean_move_time_ms
}

# Optional hard gate: disqualify anything with illegal moves or failed tests
DISQUALIFY_UNSTABLE = False

scored["raw_strength"] = (
    WEIGHTS["vs_others"] * scored["mean_vs_others"]
    + WEIGHTS["vs_baseline"] * scored["mean_vs_baseline"]
    + WEIGHTS["vs_prev"] * scored["mean_vs_prev"]
)

scored["penalty"] = (
    PENALTIES["illegal_per_move"] * scored["total_illegal"]
    + PENALTIES["failed_test"] * scored["total_tests_failed"]
    + PENALTIES["slow_ms_scale"] * scored["mean_move_time_ms"]
)

scored["composite_score"] = scored["raw_strength"] - scored["penalty"]
scored["stable"] = (scored["total_illegal"] == 0) & (scored["total_tests_failed"] == 0)

if DISQUALIFY_UNSTABLE:
    scored = scored[scored["stable"]].copy()

leaderboard = scored.sort_values("composite_score", ascending=False).reset_index(drop=True)
leaderboard.insert(0, "rank", np.arange(1, len(leaderboard) + 1))
leaderboard

,rank,model,rounds,mean_vs_others,mean_vs_baseline,mean_vs_prev,total_illegal,total_tests_failed,mean_move_time_ms,improvement_slope,raw_strength,penalty,composite_score,stable
0,1,mock_nn,5,1.00000,1.000,0.500,25,0,866.276932,-1.206738e-17,0.900000,2.673255,-1.773255,False
1,2,classical_minimax,5,0.73125,0.500,0.500,25,4,246.000302,-1.206738e-17,0.604062,3.549200,-2.945138,False
2,3,berserker1,5,0.68750,0.750,0.500,25,5,772.702469,-1.206738e-17,0.671875,3.904540,-3.232665,False
3,4,monte_carlo,5,0.25000,0.075,0.475,27,5,692.011593,1.250000e-02,0.233750,4.088402,-3.854652,False
4,5,claude_api,5,0.00000,0.000,0.000,140,0,0.000000,0.000000e+00,0.000000,14.000000,-14.000000,False


In [8]:
# Optional: inspect round-by-round trajectory
pivot = df.pivot_table(index="round", columns="model", values="win_rate_vs_previous_self", aggfunc="mean")
pivot

model,berserker1,classical_minimax,claude_api,mock_nn,monte_carlo
round,,,,,
1,0.5,0.5,0.0,0.5,0.500
2,0.5,0.5,0.0,0.5,0.375
3,0.5,0.5,0.0,0.5,0.500
4,0.5,0.5,0.0,0.5,0.500
5,0.5,0.5,0.0,0.5,0.500
